# Earth–RSA Baseline Diagnostic — Validation Notebook

Module 10. Audit-only. Compares what PyPSA-Earth retrieves by default for South Africa against what PyPSA-RSA uses (post modules 06–09 overrides).

Prerequisite: run `scripts/build_za_earth_rsa_diagnostic.py` first so the diagnostic CSVs exist.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path('.').resolve()
while not (ROOT / 'Snakefile').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print('repo root:', ROOT)

sys.path.insert(0, str(ROOT / 'scripts'))
from za_diagnostic import plots

AUDIT = ROOT / 'data' / 'za_audit'
RUN_NAME = 'za_2023_fixed_validation'
EXISTING_LINES = AUDIT / 'za_rsa_existing_lines_220kv_plus.geojson'
CLEAN_SUBS = ROOT / 'resources' / RUN_NAME / 'osm' / 'clean' / 'all_clean_substations.geojson'
ELEC_S_34 = ROOT / 'networks' / RUN_NAME / 'elec_s_34.nc'
SUPPLY_REGIONS = ROOT / 'resources' / RUN_NAME / 'bus_regions' / 'regions_onshore_elec_s_34.geojson'

fleet_df = pd.read_csv(AUDIT / 'za_ppm_vs_rsa_fleet_comparison.csv')
grid_df = pd.read_csv(AUDIT / 'za_grid_reconciliation.csv')
subs_df = pd.read_csv(AUDIT / 'za_substations_comparison.csv')
rsa_subs_df = pd.read_csv(AUDIT / 'za_rsa_substations_derived.csv')
ratings_df = pd.read_csv(AUDIT / 'za_osm_vs_stclair_ratings_comparison.csv')
ppm_only_df = pd.read_csv(AUDIT / 'za_ppm_plants_not_in_rsa.csv')
rsa_only_df = pd.read_csv(AUDIT / 'za_rsa_plants_not_in_ppm.csv')
print('loaded all diagnostic CSVs')

## 1 — Powerplants (PPM vs RSA)

Earth side: `powerplantmatching` live query for ZA. RSA side: `za_powerplant_reconciliation.csv` (229 rows). Fuzzy-matched on same carrier, ±20 km, ±30% capacity.

In [ ]:
fleet_df

In [ ]:
fig = plots.plot_fleet_capacity_by_carrier(fleet_df)
fig

In [ ]:
print('Top 20 PPM-only plants (no RSA match):')
ppm_only_df.sort_values('Capacity', ascending=False).head(20)

In [ ]:
print('Top 20 RSA-only plants (no PPM match):')
rsa_only_df.sort_values('capacity_mw_final', ascending=False).head(20)

## 2 — Transmission lines per voltage

RSA GeoJSON (324 features) bucketed by `NOMINAL_VO`. Non-220kV+ features reported as `other_kv`.

In [ ]:
grid_df

In [ ]:
plots.plot_line_count_per_voltage(grid_df)

In [ ]:
plots.plot_line_length_per_voltage(grid_df)

In [ ]:
plots.plot_network_map(ELEC_S_34, EXISTING_LINES, SUPPLY_REGIONS)

## 3 — Substations

RSA derived from union of `LINE_START` ∪ `LINE_END`. OSM filtered to ZA + voltage ≥ 220 kV.

In [ ]:
subs_df

In [ ]:
print(f'RSA derived substations: {len(rsa_subs_df)} unique')
rsa_subs_df.sort_values('n_incident_lines', ascending=False).head(20)

In [ ]:
plots.plot_substation_count_per_voltage(subs_df)

In [ ]:
plots.plot_substation_map(EXISTING_LINES, rsa_subs_df, CLEAN_SUBS)

## 4 — Line ratings (OSM s_nom vs St Clair N-1)

Per-corridor sum of `n.lines.s_nom` from `elec_s_34.nc` vs `st_clair_n1_mw` from the 65-corridor table.

In [ ]:
ratings_main = ratings_df[ratings_df['bus0'] != '_summary_']
ratings_summary = ratings_df[ratings_df['bus0'] == '_summary_']
print('Per-direction summary:')
ratings_summary

In [ ]:
plots.plot_ratings_ratio_distribution(ratings_df)

In [ ]:
plots.plot_ratings_scatter(ratings_df)

In [ ]:
print('Top 10 most OVER-rated corridors (OSM >> St Clair):')
ratings_main.sort_values('ratio_osm_to_stclair', ascending=False).head(10)

In [ ]:
print('Top 10 most UNDER-rated corridors (OSM << St Clair):')
valid = ratings_main[ratings_main['ratio_osm_to_stclair'].notna()]
valid.sort_values('ratio_osm_to_stclair', ascending=True).head(10)

## 5 — Synthesis

Quick verdict per dimension.

In [ ]:
summary = []
delta_total = fleet_df['delta_mw'].sum()
summary.append(('Fleet', f'PPM-RSA delta {delta_total:+.0f} MW',
                'over' if delta_total > 0 else 'under'))
g = grid_df[grid_df['voltage_bucket'].isin(['220kV','275kV','400kV','765kV'])]
osm_len = g['osm_length_km'].fillna(0).sum()
rsa_len = g['rsa_length_km'].fillna(0).sum()
summary.append(('Line length (220kV+)',
                f'OSM {osm_len:.0f} km vs RSA {rsa_len:.0f} km',
                f'coverage {osm_len/rsa_len:.2f}'))
subs_total = subs_df[subs_df['voltage_bucket'] == '220kv_plus_total'].iloc[0]
summary.append(('Substations (220kV+)',
                f"OSM {subs_total['osm_substation_count']} vs RSA {subs_total['rsa_substation_count']}",
                f"ratio {subs_total['osm_coverage_ratio']}"))
n_over = (ratings_main['direction'] == 'osm_over').sum()
n_under = (ratings_main['direction'] == 'osm_under').sum()
n_within = (ratings_main['direction'] == 'within_20pct').sum()
summary.append(('Line ratings (65 corridors)',
                f'over={n_over}, under={n_under}, within={n_within}', ''))
pd.DataFrame(summary, columns=['Dimension', 'Numbers', 'Verdict'])